![Ecommerce Sales Cover](ecommerce_sales_cover.png)

# Sales Forecasting and Market-Basket Analysis for a Kenyan Baby Products Retailer

This project predicts future sales and uncovers which products are bought together, using 36 months of data from a Kenyan baby-products retailer. The results help the business plan stock and design bundles that increase order value.

### 1.Business Understanding

### 1.1 Background

The client is a Kenyan retailer of baby products that sells through three channels: a physical point of sale, an online store and draft orders. The dataset covers 36 months of trading, from November 2020 to October 2023, and contains 28,755 transaction lines recorded in Kenyan shillings (KES). Over this period the business grew quickly, with yearly order counts rising from about 3,000 in 2021 to over 5,700 in the first ten months of 2023, and monthly net sales climbing from under KES 1 million to a peak of roughly KES 4.4 million.

This growth makes planning harder. Stock, cash and promotions are still decided largely from experience, so the retailer risks running out of fast-moving items such as formula and diapers, or tying up money in slow-moving ones. The retailer also does not know which products customers tend to buy together, even though about two in three orders contain more than one distinct product.

### 1.2 Problem Statement

The retailer plans stock, cash and promotions mostly from experience, without a forecast of future demand and without knowing which products customers buy together. This leads to two costly problems: popular items such as formula and diapers can run out while slow-moving items tie up money, and chances to raise order value through bundles and add-on offers are missed.

> **How can past transaction data be used to forecast monthly sales and to identify the products customers frequently buy together?**

### 1.3 Business Objectives

1. **Plan stock and cash better.** Know roughly how much the business will sell in the coming months, so it can order the right quantities and avoid stock-outs and overstocking.
2. **Increase the value of each order.** Find products that are often bought together, so the retailer can offer bundles, place related items near each other and suggest add-ons.
3. **Make the results easy to use.** Give the owner outputs that can be read and applied without technical knowledge.

### 1.4 Data Science Objectives

Each business objective is turned into a specific analysis task.

| Business objective | Data science task | Method |
|---|---|---|
| Plan stock and cash better | Forecast monthly net sales (total, then by product category) for the next 3 months | Time-series forecasting, starting from a seasonal-naive baseline |
| Increase the value of each order | Find product and category combinations that appear together in orders more often than chance | Market-basket analysis (association rules with Apriori or FP-Growth) |
| Make the results easy to use | Save the final model and rules and provide a simple function and report | Model persistence and a lightweight app |

### 1.5 Success Criteria (Metrics of Success)

These targets are set before modeling, so the results are judged fairly in the Evaluation phase.

**Sales forecasting**
- **Primary metric:** WAPE (weighted absolute percentage error), measured on the last 3 months held out from training.
- **Target:** WAPE of 15% or lower for total monthly sales.
- **Baseline to beat:** a seasonal-naive forecast. The chosen model should improve on it by at least 10%.

**Market-basket analysis**
- **Support** of at least 0.5% of orders, so rules are not based on a handful of orders.
- **Confidence** of at least 20% and **lift** of at least 1.2, meaning the products sell together at least 20% more often than chance.
- **Usefulness:** at least 10 rules that the retailer considers worth acting on.

These thresholds are starting points and may be adjusted once we see how sparse the baskets are.

### 1.6 Scope and Assumptions

**In scope**
- Monthly sales forecasting for total net sales and by product category.
- Market-basket analysis at order level, using products and categories.
- Transactions from November 2020 to October 2023, in Kenyan shillings (KES).

**Out of scope**
- Customer-level analysis, because the data has no customer ID.
- Daily forecasts for individual products, because the data is too sparse.
- Pricing optimization.

**Assumptions**
- One order (`order_name`) represents one shopping basket.
- Net sales is the revenue measure to forecast.
- Delivery charges, gift cards and other non-product lines are not products and will be excluded from the basket analysis.

### 1.7 Risks and Constraints

| Risk | Why it matters | How we will handle it |
|---|---|---|
| Only 36 months of history | Seasonality is hard to estimate from three yearly cycles | Use simple, robust models and report forecast uncertainty |
| Strong growth over the period | Models that ignore trend will underpredict | Use methods that handle trend, such as Holt-Winters or SARIMA |
| Delivery "Rider" charges recorded as products | They would create meaningless product pairings | Identify and remove them in Data Preparation |
| Inconsistent product names and many missing categories | Pairs get split across spellings and rules weaken | Standardize names and build a product-to-category mapping |
| Timestamps stored in UTC | Late-evening orders could fall into the wrong month | Convert to Nairobi time (UTC+3) before grouping |
| No customer ID | Customers cannot be segmented | Analyse at order level only |

### 1.8 Project Plan

| Phase | Main work | Output |
|---|---|---|
| 2. Data Understanding | Load data, explore it, check quality | Data profile and EDA findings |
| 3. Data Preparation | Clean products, convert time zone, build monthly series and basket table | Analysis-ready datasets |
| 4. Modeling | Baseline first, then tuned forecasting models and association rules | Trained models and rules |
| 5. Evaluation | Compare results with the success criteria in 1.5 | Pass or fail assessment and recommendations |
| 6. Deployment | Save the model, build a reusable pipeline and a simple app | Working tool for the retailer |

In [13]:
# core libraries and notebook display settings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import os
from dotenv import load_dotenv
from sqlalchemy import create_engine

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 150)
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (12, 5)


In [14]:
load_dotenv()

PGUSER = os.getenv("PGUSER")
PGPASSWORD = os.getenv("PGPASSWORD")
PGHOST = os.getenv("PGHOST")
PGPORT = os.getenv("PGPORT")
PGDATABASE = os.getenv("PGDATABASE")

# Build the connection string
DATABASE_URL = f"postgresql+psycopg2://{PGUSER}:{PGPASSWORD}@{PGHOST}:{PGPORT}/{PGDATABASE}?sslmode=require"

engine = create_engine(DATABASE_URL)
print("Connected to cloud database (Neon) — engine ready!")

Connected to cloud database (Neon) — engine ready!


In [15]:

load_dotenv(override=True)

neon_url = os.getenv("NEON_URL").replace("postgresql://", "postgresql+psycopg2://", 1)
engine = create_engine(neon_url)

df = pd.read_sql("SELECT * FROM sales", con=engine)
print(df.shape)   # expect (28755, 17)
df.head()

(28755, 17)


,sale_id,order_id,date,order_name,transaction_type,sale_type,sales_channel,product_type,product,net_quantity,gross_sales,discounts,returns,net_sales,shipping,taxes,total_sales
0,8725409595544,2854884835480,2020-11-10 22:39:33+00:00,TS1011,product,order,Online Store,Bathing & Skin Care,Mamia Sensitive Bathing & Skin Care 64 Bathing...,1,200.0,0.0,0.0,200.0,0.0,0.0,200.0
1,8752046604440,2864488448152,2020-11-16 13:43:01+00:00,TS1012,product,order,Point of Sale,Baby Formula,NANNYcare Growing Up Milk Goat Milk Based 3 Fr...,2,9000.0,0.0,0.0,9000.0,0.0,0.0,9000.0
2,8752055845016,2864491954328,2020-11-16 13:45:58+00:00,TS1013,product,order,Point of Sale,Baby Diapers,DryNites Spider-Man Baby & Toddler Clothing 16pk,1,1800.0,0.0,0.0,1800.0,0.0,0.0,1800.0
3,8752055877784,2864491954328,2020-11-16 13:45:58+00:00,TS1013,product,order,Point of Sale,Bathing & Skin Care,Metanium Nappy Rash Ointment 30g,1,1500.0,0.0,0.0,1500.0,0.0,0.0,1500.0
4,8752070656152,2864497066136,2020-11-16 13:49:47+00:00,TS1014,product,order,Point of Sale,Baby Formula,Aptamil 1 First Baby Milk Formula from Birth 800g,1,3000.0,0.0,0.0,3000.0,0.0,0.0,3000.0


In [16]:
SOURCE = "Local"    # change to "Neon" for the cloud database

if SOURCE == "Local":
    url = (f"postgresql+psycopg2://{os.getenv('PGUSER')}:{os.getenv('PGPASSWORD')}"
           f"@{os.getenv('PGHOST')}:{os.getenv('PGPORT')}/{os.getenv('PGDATABASE')}?sslmode=prefer")
else:
    url = os.getenv("NEON_URL").replace("postgresql://", "postgresql+psycopg2://", 1)

engine = create_engine(url)
df = pd.read_sql("SELECT * FROM sales", con=engine)
print(SOURCE, df.shape)

Local (28755, 17)
